# Task 2 - Recommendation System Development

Load and integrate the allocated customer, product and transaction data. Build and evaluate a popularity-based recommender plus one additional method. Keep business interpretation for Task 3.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_DIR = Path('../data')
customers = pd.read_csv(DATA_DIR / 'Customers_repeat.csv')
products = pd.read_csv(DATA_DIR / 'Products_sort.csv')
transactions = pd.read_csv(DATA_DIR / 'Transactions_made.csv')
customers.head(), products.head(), transactions.head()

(   CustomerID AgeGroup Region LoyaltyLevel  MembershipYears
 0           1    18-25  North       Bronze                1
 1           2    18-25   East       Bronze                2
 2           3      50+   East       Bronze                5
 3           4    36-50   East       Silver               12
 4           5    26-35   East       Silver               13,
    ProductID     Category       Brand   Price  Rating  PopularityScore
 0          1      Kitchen     FitLife  460.84    4.46                2
 1          2  Electronics     TechMax  211.35    3.39                1
 2          3         Home     HomePro  337.91    3.31                2
 3          4       Gaming     TechMax  323.63    3.66                1
 4          5      kitchen  PrimeGoods  202.09    2.52                4,
    CustomerID  ProductID  PurchaseCount  LastPurchaseDays    Spend
 0           1        120              2                92    36.24
 1           2         55              4               293  1872

In [3]:
# Validate keys and missing values before joining tables.
for name, frame in {'customers': customers, 'products': products, 'transactions': transactions}.items():
    print(name, frame.shape)
    display(frame.isna().sum().sort_values(ascending=False).head())
print('customer IDs missing from customer table:', (~transactions['CustomerID'].isin(customers['CustomerID'])).sum())
print('product IDs missing from product table:', (~transactions['ProductID'].isin(products['ProductID'])).sum())
interaction_matrix = transactions.pivot_table(index='CustomerID', columns='ProductID', values='PurchaseCount', aggfunc='sum', fill_value=0)
interaction_matrix.shape

customers (1000, 5)


CustomerID         0
AgeGroup           0
Region             0
LoyaltyLevel       0
MembershipYears    0
dtype: int64

products (200, 6)


Rating       15
ProductID     0
Category      0
Brand         0
Price         0
dtype: int64

transactions (18711, 5)


CustomerID          0
ProductID           0
PurchaseCount       0
LastPurchaseDays    0
Spend               0
dtype: int64

customer IDs missing from customer table: 0
product IDs missing from product table: 0


(1000, 200)

## First three Task 2 requirements

The tables are joined using their customer and product IDs. Customer value is then summarised using total spend and total purchase frequency.

In [4]:
# Integrate the three tables.
integrated_data = transactions.merge(customers, on='CustomerID', how='left')
integrated_data = integrated_data.merge(products, on='ProductID', how='left')

print('Integrated rows and columns:', integrated_data.shape)
print('Missing customer details:', integrated_data['AgeGroup'].isna().sum())
print('Missing product details:', integrated_data['Category'].isna().sum())

# Top 10 customers by spend and purchase frequency.
customer_summary = transactions.groupby('CustomerID').agg(
    TotalSpend=('Spend', 'sum'),
    TotalPurchases=('PurchaseCount', 'sum')
).sort_values('TotalSpend', ascending=False)

display(customer_summary.head(10))
display(customer_summary.sort_values('TotalPurchases', ascending=False).head(10))

Integrated rows and columns: (18711, 14)
Missing customer details: 0
Missing product details: 0


,TotalSpend,TotalPurchases
CustomerID,,
971,32222.52,83
174,29802.65,78
227,28402.94,81
184,28213.86,80
761,27805.10,77
611,27618.73,77
840,27082.53,81
434,26933.14,68
880,26651.43,71


,TotalSpend,TotalPurchases
CustomerID,,
79,26046.02,89
95,19797.80,87
553,9573.32,87
373,25882.32,85
831,17269.04,84
265,8317.85,83
971,32222.52,83
340,9609.71,82
448,13576.84,82


## Required evidence checklist
- [x] Data quality checks and integrated customer-product-transaction data
- [x] Customer-product interaction matrix
- [x] Top 10 customers by spend and purchase frequency
- [ ] Top 10 products by revenue, category revenue and brand revenue
- [ ] Three groupby analyses and two pivot tables
- [ ] Popularity-based recommender
- [ ] Second recommendation approach
- [ ] Evaluation and examples for at least five customers